# 01 · CaRS-50 — build the pool

*Swales CARS moves in research-article introductions (3 or 11 classes)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** 50 BioRxiv article introductions, annotated sentence by sentence with Swales' CARS Move and Step scheme. **The annotators themselves reached only κ ≈ 0.43** — so on this track, "the model is wrong" and "the scheme is fuzzy" are both live explanations, and telling them apart is the interesting part.

**Difficulty of the labeling judgment:** ★★★ — hard. Judging moves in an introduction needs more context than a single sentence gives you.

**Licence:** CC BY 4.0  
**Cite:** Lam, C. & Nnamoko, N. (2025). *Mendeley Data*, V1. doi:10.17632/kwr9s5c4nk.1

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

Those three keys are required on every track. Two tracks add more: `cars50` and `raamove` ask what a sentence *does in a passage*, which is not always decidable from the sentence on its own, so their items also carry `doc_id`, `sent_index`, `n_sents` and `context`. Extra keys are safe everywhere — nothing in the pipeline checks for keys it does not need.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first.
# ------------------------------------------------------------------
# Finds your group's shared folder in Google Drive. Everything this project
# keeps goes in there: a Colab runtime is wiped when it resets, and nobody
# else in your group can see inside it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work in the RUNTIME, not in Drive: the next cells download a whole
    # corpus, and raw data is big, mostly not ours to redistribute, and one
    # command to fetch again. The pool you build from it is what persists.
    os.makedirs("/content/raw", exist_ok=True)
    os.chdir("/content/raw")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

from config import *      # TRACK, GROUP, SEED, N_PER_CLASS, and every path

describe()                  # what this notebook is working on


## Step 1 — Download the raw data

This one is on **Mendeley Data**, which has a public API. We ask it for the dataset's file list, then download each file. The CDN refuses requests that do not look like a browser, hence the `User-Agent` header.

In [ ]:
import json, urllib.request, pathlib

RAW_DIR = pathlib.Path("cars50")
RAW_DIR.mkdir(exist_ok=True)

def fetch(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(request, timeout=60)

meta = json.loads(fetch("https://data.mendeley.com/public-api/datasets/kwr9s5c4nk").read())
for record in meta["files"]:
    target = RAW_DIR / record["filename"]
    if not target.exists():
        target.write_bytes(fetch(record["content_details"]["download_url"]).read())
print("downloaded", len(list(RAW_DIR.glob("*.xml"))), "XML files")

## Step 2 — Look at the raw format

**XML** this time. Each sentence carries a `step` code like `1b`:

```xml
<sentence><sentenceID/><text/><step>1b</step></sentence>
```

In [ ]:
print(open(sorted(RAW_DIR.glob("*.xml"))[0], encoding="utf-8").read()[:900])

### Reading XML with `ElementTree`

XML is nested, so unlike a TSV or a CSV you cannot get at a field by position. Python's built-in `xml.etree.ElementTree` parses the file into a tree of elements you then navigate by tag name.

**The nesting, outermost first:**

* `<biology_intro>` — the root element, one per file
* `<fulltext>` — the introduction itself
* `<paragraph>` — one or more per introduction
* `<sentence>` — one or more per paragraph, each carrying:
  * `<sentenceID>` — an identifier
  * `<text>` — the sentence
  * `<step>` — the rhetorical step code, e.g. `1b`

The reshaping function below uses exactly these:

1. `ET.parse(path)` — read one file into a tree.
2. `tree.iter("sentence")` — every `<sentence>` at **any** depth, so you never have to walk the paragraphs yourself. It yields them in document order, which is what lets each sentence keep its place in the introduction.
3. `element.find("text")` — the first child with that tag, or `None` if it is missing. That `None` is why the reshaping code checks before using it.
4. `element.text` — the string inside a tag. It is `None` for an empty tag, hence the `(… or "")` guard before `.strip()`.
5. `path.stem` — the filename without `.xml`, e.g. `text001`. That is the document id. **Not** `<sentenceID>`: those are padded three different ways, mix two widths inside `text038.xml`, and `t025s020` appears twice in `text025.xml`. Position comes from counting, not from reading an id.

In [ ]:
import xml.etree.ElementTree as ET

first_file = sorted(RAW_DIR.glob("*.xml"))[0]
tree = ET.parse(first_file)

print("reading", first_file.name, "\n")
for i, sentence in enumerate(tree.iter("sentence")):
    if i >= 3:                       # just the first three, to keep the output short
        break
    for tag in ("sentenceID", "text", "step"):
        child = sentence.find(tag)
        print(" ", tag + ":", child.text.strip() if child is not None and child.text else "MISSING")
    print()

## Step 3 — Reshape into the canonical schema

The parsing is written for you, and it gives you **both granularities at once**:

- the leading digit of `1b` is the **Move** → 3 classes;
- the whole code `1b` is the **Step** → 11 classes.

Sentences with no code, or a code that does not start with a move digit, are dropped either way. In *this* corpus that guard never actually fires — all 1297 sentences are coded — so do not write "we dropped N malformed sentences" in your report without checking the number first. The guard is there because the next corpus you meet will need it.

✏️ **Which one you study is the decision**, and on this track it is the whole shape of the project. Three classes with a few hundred items each is a fair task you can sample 40 items from comfortably. Eleven classes over the same sentences means some steps have barely a dozen examples, a confusion matrix with 121 cells, and an annotation job your two coders will find genuinely hard — remember the original annotators managed only κ ≈ 0.43 at this granularity.

Neither is the safe answer. The 11-class version makes a better project **if** you have the time to annotate it properly and the nerve to report a low F1 with a good explanation. Decide now, write it in `PLAN.md`, and do not switch after you have seen the numbers.

### Each sentence keeps its introduction

The difficulty note at the top of this notebook says judging a move needs more context than a single sentence gives you. So the code below does not throw the introduction away. It reads each file **twice** — once to collect the whole introduction in order, once to emit the items — and every item comes out carrying four extra fields on top of the canonical three:

| field | what it is |
|---|---|
| `doc_id` | which introduction, e.g. `text001` |
| `sent_index` | where in it, counting from 0 |
| `n_sents` | how many sentences the introduction has |
| `context` | the introduction itself, one sentence per line |

One detail worth arguing about: `context` keeps **every** sentence that has text, including the ones dropped for having no usable code. They belong there because a reader saw them — filtering them out would hand the model a doctored introduction that never existed. Both granularities get identical fields; it is the same sentence in the same passage, just labelled two ways.

Those fields travel with the item all the way: notebook 03 shows the introduction to your two coders, and notebook 04 can put it in the prompt. `prompts/cars50.txt` shows the model the sentence alone, `prompts/cars50_context.txt` shows it the introduction first. These are 26 sentences on average and up to 47, so the context condition is noticeably slower to run — worth knowing before you start it at 4pm.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1

    for item in items:
        ### Copy before writing ###
        new_item = dict(item)                    # Work on a copy, so the caller's item is left alone.

        ### Stamp the id ###
        new_item["id"] = next_id                 # Overwrite whatever id was there with the running number.
        renumbered.append(new_item)              # Keep it in the order it arrived.
        next_id = next_id + 1                    # Advance, so the next item gets a fresh id.

    return renumbered

def reshape_cars50(cars50_dir):
    """Parse the 50 XML introductions into TWO datasets: moves, and move+step.

    XML shape:
        <sentence><sentenceID/><text/><step>1b</step></sentence>

    The `step` code is like "1b": the leading DIGIT is the Move, the whole code is the
    Step. So one parse gives two granularities, and which you use is a scheme decision:
    3 classes is a fair task, 11 classes is the stretch version. Returns
    (move_rows, step_rows).

    A move is a rhetorical function WITHIN an introduction, so each item also carries the
    introduction it came from - see the note on the two-pass loop below.
    """
    source_dir = Path(cars50_dir)
    move_rows = []
    step_rows = []

    ### Walk the 50 XML files ###
    for xml_path in sorted(source_dir.glob("*.xml")):   # One file per article introduction.

        ### Parse one file into a tree ###
        tree = ET.parse(xml_path)                # ElementTree turns the XML into a navigable tree.
        doc_id = xml_path.stem                   # e.g. "text001". The FILENAME - the <sentenceID> tags are not reliable.

        ### PASS 1: read the whole introduction, in order ###
        # Every sentence that has text goes in here, including ones we are about to drop
        # for having no usable code. They belong in the passage because a reader saw them:
        # leaving them out would hand the model a doctored introduction.
        passage = []
        for sentence in tree.iter("sentence"):   # .iter() finds them at any depth, so the paragraph nesting does not matter.
            text_element = sentence.find("text")     # The <text> child, or None if absent.
            step_element = sentence.find("step")     # The <step> child, or None if absent.
            text = (text_element.text or "").strip() if text_element is not None else ""   # `or ""` guards an empty tag, whose .text is None.
            if text:
                passage.append((text, step_element))

        texts = [text for text, _ in passage]    # Just the sentences.
        context = "\n".join(texts)               # One string, newlines kept so the sentences stay visible.

        ### PASS 2: emit an item for each sentence that carries a usable code ###
        # Position comes from enumerate(), never from <sentenceID>: those ids are padded
        # three different ways, mix two widths inside text038.xml, and t025s020 appears
        # twice in text025.xml.
        for position, (text, step_element) in enumerate(passage):
            code = (step_element.text or "").strip() if step_element is not None else ""   # e.g. "1b".

            # Skip anything unlabelled, or whose code does not start with a move digit.
            if not code or not code[0].isdigit():
                continue

            ### Where this sentence sits - the same for both granularities ###
            where = {"doc_id": doc_id,           # Which introduction this sentence is from.
                     "sent_index": position,     # Where in it - 0 is the first sentence.
                     "n_sents": len(texts),      # How long the introduction is.
                     "context": context}         # The introduction itself.

            ### Record the SAME sentence at both granularities ###
            move_rows.append({"id": 0, "text": text,
                              "label": "Move " + code[0], **where})   # Leading digit only -> 3 classes.
            step_rows.append({"id": 0, "text": text,
                              "label": code, **where})                # Whole code -> 11 classes.

    return reid(move_rows), reid(step_rows)      # Two datasets, each with ids running 1..N.

In [ ]:
move_rows, step_rows = reshape_cars50(RAW_DIR)
print("moves:", len(move_rows), " steps:", len(step_rows))

from collections import Counter
print("move classes:", Counter(r["label"] for r in move_rows))
print("step classes:", Counter(r["label"] for r in step_rows))

In [ ]:
# ✏️ Step 3a · Choose your granularity ───────────────────────────
# Goal      : pick the version of the scheme your group will actually study.
# Shape     : rows = move_rows    # 3 classes
#             rows = step_rows    # 11 classes
# Produce   : rows (a list) — either move_rows or step_rows      ← later cells use this name
# Note      : one line. Spend the time on the ARGUMENT, not the typing —
#             PLAN.md asks you to justify it in a sentence.
# Careful   : whichever you pick, the label names in your prompt and your
#             annotation sheet must match these exactly ("Move 1", or
#             "1b").

# ✏️ your code here


## Step 4 — Check the label balance

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
print("fields per item:", list(rows[0]))

# Peek at the first three. `context`, where a track has one, is trimmed: it is
# the whole passage and would bury everything else in this output.
for item in rows[:3]:
    preview = dict(item)
    if preview.get("context"):
        preview["context"] = preview["context"][:70] + " …"
    print(preview)

## Step 5 — Save it

In [ ]:
# Save the pool into your group's Drive folder, under the exact name notebook
# 02 will look for. POOL_PATH comes from config.py, so the two cannot drift.
import json

if TRACK not in ['cars50', 'cars50_step']:
    raise RuntimeError(
        "config.py says TRACK = " + repr(TRACK) + ", but this is the cars50 "
        "notebook. POOL_PATH points at " + POOL_PATH.name + ", so saving now "
        "would put cars50 data in another track's file. Set TRACK to one "
        "of cars50 · cars50_step in config.py, then re-run the SETUP cell.")

POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(POOL_PATH, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", POOL_PATH)

# If you chose steps, set TRACK = "cars50_step" in config.py before running this: POOL_PATH then becomes cars50_step_pool.json, and the finer scheme gets its own file rather than overwriting the 3-move one. You will need prompts/cars50_step.txt too — copy cars50.txt and rewrite the labels.

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** open `02_sample.ipynb`. It reads `POOL_PATH` — the file the cell above just wrote, in your group's Drive folder. Nothing to copy, nothing to paste: that path is the handoff, and both notebooks get it from the same `config.py`.